# Exploratory Data Analysis - Heart Disease Prediction

This notebook performs exploratory data analysis on the heart disease dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/pkiohd.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nDataset Info:")
print(df.info())

In [ ]:
# Display first few rows
df.head()

## 2. Statistical Summary

In [ ]:
# Statistical summary
df.describe()

## 3. Missing Values Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print("Missing Values:")
    print(missing_df)
else:
    print("No missing values found.")

## 4. Target Variable Distribution

In [ ]:
# Target column (adjust based on your dataset)
target_col = 'target'  # Change this to your target column name

if target_col in df.columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x=target_col)
    plt.title('Target Variable Distribution')
    plt.xlabel('Heart Disease')
    plt.ylabel('Count')
    plt.show()
    
    # Print class distribution
    print("\nClass Distribution:")
    print(df[target_col].value_counts())
    print(f"\nClass Balance: {df[target_col].value_counts(normalize=True)}")
else:
    print(f"Target column '{target_col}' not found. Please specify the correct target column.")

## 5. Feature Distributions

In [ ]:
# Numerical features distribution
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if target_col in numerical_cols:
    numerical_cols.remove(target_col)

n_cols = 4
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes]

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'Distribution of {col}')

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
plt.figure(figsize=(14, 12))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
if target_col in df.columns:
    target_corr = df[numerical_cols + [target_col]].corr()[target_col].sort_values(ascending=False)
    
    plt.figure(figsize=(10, 8))
    target_corr.drop(target_col).plot(kind='barh')
    plt.xlabel('Correlation Coefficient')
    plt.ylabel('Features')
    plt.title(f'Feature Correlation with {target_col}')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 7. Box Plots for Outlier Detection

In [ ]:
n_cols = 4
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes]

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        axes[idx].boxplot(df[col].dropna())
        axes[idx].set_title(f'Box Plot: {col}')
        axes[idx].set_ylabel(col)

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 8. Pair Plot of Key Features

In [ ]:
# Select top correlated features for pair plot
if target_col in df.columns:
    top_features = target_corr.abs().nlargest(6).index.tolist()
    if target_col in top_features:
        top_features.remove(target_col)
    
    if len(top_features) > 0:
        pair_df = df[top_features + [target_col]]
        sns.pairplot(pair_df, hue=target_col, diag_kind='kde')
        plt.suptitle('Pair Plot of Key Features', y=1.02)
        plt.show()

## 9. Categorical Features Analysis

In [ ]:
# Categorical features
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_cols) > 0:
    n_cols = 3
    n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for idx, col in enumerate(categorical_cols):
        if idx < len(axes):
            df[col].value_counts().plot(kind='bar', ax=axes[idx])
            axes[idx].set_title(f'Distribution of {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Count')
            axes[idx].tick_params(axis='x', rotation=45)
    
    # Hide unused subplots
    for idx in range(len(categorical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found.")

## 10. Feature vs Target Analysis

In [ ]:
if target_col in df.columns:
    n_cols = 4
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            for target_val in df[target_col].unique():
                sns.kdeplot(df[df[target_col] == target_val][col].dropna(), 
                           label=f'{target_col}={target_val}', ax=axes[idx])
            axes[idx].set_title(f'Distribution of {col} by {target_col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Density')
            axes[idx].legend()
    
    # Hide unused subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 11. Summary Statistics by Target Class

In [ ]:
if target_col in df.columns:
    print("Summary Statistics by Target Class:")
    print(df.groupby(target_col)[numerical_cols].describe().transpose())

## 12. Data Quality Report

In [ ]:
print("="*60)
print("DATA QUALITY REPORT")
print("="*60)
print(f"Total Samples: {len(df)}")
print(f"Total Features: {len(df.columns)}")
print(f"Numerical Features: {len(numerical_cols)}")
print(f"Categorical Features: {len(categorical_cols)}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print("="*60)